## 1. Entradas — edite apenas esta célula

In [ ]:
# ── Parâmetros fixos ────────────────────────────────────────────────────
epsilon     = 0.54                   # [-]  fração estrutural = Ms / M0
                                     #      (0.54 equivale à planilha original)
                                     #      valores típicos: 0.20 – 0.55
c0          = 2138.58                # [m/s] velocidade de exaustão (Isp × g)
d_cm        = 18.0                   # [cm]  diâmetro da fuselagem
Au          = 0.015751955874864804   # [m²]  área de saída do bocal
alfa0       = 85.0                   # [°]   ângulo de lançamento
H_target_km = 20.0                   # [km]  altitude alvo

# Range da curva T_sl × tb
Tsl_min_kN = 1.0
Tsl_max_kN = 4.0
n_pontos   = 30


## 2. Física — não editar

In [ ]:
import numpy as np, math, matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq

PI=math.pi; ATM=101300.0; g=9.81; R=287.0; GAMMA=1.4; rad=PI/180; k=g/R
Temp0=288.15; lam1=-0.0065; Temp1=216.65; Temp2=Temp1
lam3=0.001; Temp3=Temp2+lam3*12000; lam4=0.0028; Temp4=Temp3+lam4*15000

def Tz(z):
    if   z<=11000: return Temp0+lam1*z
    elif z<=20000: return Temp1
    elif z<=32000: return Temp2+lam3*(z-20000)
    elif z<=47000: return Temp3+lam4*(z-32000)
    else:          return Temp4

def pdiz(z):
    if z<=11000:   pz=(Tz(z)/Temp0)**(-k/lam1)
    elif z<=20000: A=(1/lam1)*math.log(Temp1/Temp0);            pz=math.exp(-k*(A+(z-11000)/Temp1))
    elif z<=32000: A=(1/lam1)*math.log(Temp1/Temp0)+9000/Temp1; pz=math.exp(-k*(A+math.log(Tz(z)/Temp2)/lam3))
    elif z<=47000: A=(1/lam1)*math.log(Temp1/Temp0)+9000/Temp1+(1/lam3)*math.log(Temp3/Temp2); pz=math.exp(-k*(A+math.log(Tz(z)/Temp3)/lam4))
    else:          pz=1e-10
    return pz*ATM

def rodiz(z): return pdiz(z)/Tz(z)/R

_M=np.array([0.0834,0.4964,0.875,0.9438,0.9954,1.0297,1.0813,1.1329,1.3738,1.6492,
             1.8557,2.0795,2.3204,2.5786,2.8712,3.1638,3.4736,3.8178,4.1792,4.5406,
             4.9193,5.2807,5.6765,6.0895,6.5025,6.9156,7.3114,7.7244,8.1374,8.5504,
             8.9635,9.3937,9.8067,10.2025,10.736,11.1662,11.5792,12.0095,12.4225])
_C=np.array([0.40382,0.40385,0.40735,0.43164,0.47502,0.50972,0.55310,0.59648,0.61905,
             0.58784,0.54969,0.51327,0.47859,0.44391,0.41097,0.38150,0.35376,0.32950,
             0.30697,0.28618,0.26886,0.25501,0.24116,0.23078,0.22214,0.21523,0.21006,
             0.20141,0.19624,0.18933,0.18416,0.18072,0.17728,0.17211,0.16868,0.16698,
             0.16354,0.16184,0.16013])
_spl=CubicSpline(_M,_C,bc_type='natural')
def splineV(m): return float(_spl(m))

# ── Integrador Heun dt=0.01s (exato do VBA) ─────────────────────────────
def run_trajectory(ms, tb, C2, save_every=50):
    """
    Retorna dict com arrays (t, h, V, Mach, Drag, alfa, Cd, ag)
    e escalares (y_bo, V_bo, alfa_bo, y_apo, x_apo).
    save_every=50 => salva a cada 0.5s (como a planilha).
    """
    DT=0.01; f=C2; F0=f*ms*c0/tb
    Aref=PI/4*(d_cm/100)**2
    t=0.0; ya=0.0; xa=0.0; alfaa=alfa0; Va=0.0
    ts=[]; hs=[]; Vs=[]; Machs=[]; Drags=[]; alfas=[]; Cds=[]; ags=[]; xs=[]
    step=0

    # Boost
    while t<=tb:
        frac=1.0+f*(1.0-t/tb)
        ro=rodiz(ya); Ma=Va/math.sqrt(GAMMA*R*Tz(ya)) if Va>0 else 0
        Cda=splineV(Ma); dP=ATM-pdiz(ya); c=c0*(1.0+Au*dP/F0)
        aga=c*f/(tb*frac*g)-0.5*ro*Va**2*(Aref*Cda/ms/g/frac)-math.sin(alfaa*rad)
        dVdt=aga*g; dadt=-math.cos(alfaa*rad)*g/Va if Va!=0 else 0
        dydt=Va*math.sin(alfaa*rad); dxdt=Va*math.cos(alfaa*rad)
        V1=Va+dVdt*DT; a1=alfaa+dadt*DT; y1=max(ya+dydt*DT,0); x1=xa+dxdt*DT
        ro=rodiz(y1); Ma1=V1/math.sqrt(GAMMA*R*Tz(y1)) if V1>0 else 0
        Cd1=splineV(Ma1); dP=ATM-pdiz(y1); c=c0*(1.0+Au*dP/F0)
        ag1=c*f/(tb*frac*g)-0.5*ro*V1**2*(Aref*Cd1/ms/g/frac)-math.sin(a1*rad)
        dVdt1=ag1*g; dadt1=-math.cos(a1*rad)*g/V1 if V1!=0 else 0
        Va+=0.5*(dVdt+dVdt1)*DT; alfaa+=0.5*(dadt+dadt1)*DT
        ya+=0.5*(dydt+V1*math.sin(a1*rad))*DT; xa+=0.5*(dxdt+V1*math.cos(a1*rad))*DT
        if step%save_every==0:
            ro2=rodiz(ya); Ma2=Va/math.sqrt(GAMMA*R*Tz(ya)) if Va>0 else 0
            Cd2=splineV(Ma2)
            ts.append(t); hs.append(ya); Vs.append(Va); Machs.append(Ma2)
            Drags.append(0.5*ro2*Cd2*Aref*Va**2); alfas.append(alfaa)
            Cds.append(Cd2); ags.append(aga); xs.append(xa)
        t+=DT; step+=1

    y_bo=ya; V_bo=Va; alfa_bo=alfaa; x_bo=xa

    # Coast
    while Va>=0:
        ro=rodiz(ya); Ma=Va/math.sqrt(GAMMA*R*Tz(ya)) if Va>0 else 0
        Cda=splineV(Ma); aga=-0.5*ro*Va**2*(Aref*Cda/ms/g)-math.sin(alfaa*rad)
        dVdt=aga*g; dadt=-math.cos(alfaa*rad)*g/Va if Va!=0 else 0
        dydt=Va*math.sin(alfaa*rad); dxdt=Va*math.cos(alfaa*rad)
        V1=Va+dVdt*DT; a1=alfaa+dadt*DT; y1=ya+dydt*DT; x1=xa+dxdt*DT
        ro=rodiz(y1); Ma1=V1/math.sqrt(GAMMA*R*Tz(y1)) if V1>0 else 0
        Cd1=splineV(Ma1); ag1=-0.5*ro*V1**2*(Aref*Cd1/ms/g)-math.sin(a1*rad)
        dVdt1=ag1*g; dadt1=-math.cos(a1*rad)*g/V1 if V1!=0 else 0
        Va+=0.5*(dVdt+dVdt1)*DT; alfaa+=0.5*(dadt+dadt1)*DT
        ya+=0.5*(dydt+V1*math.sin(a1*rad))*DT; xa+=0.5*(dxdt+V1*math.cos(a1*rad))*DT
        if step%save_every==0:
            ro2=rodiz(ya); Ma2=Va/math.sqrt(GAMMA*R*Tz(ya)) if Va>0 else 0
            Cd2=splineV(Ma2)
            ts.append(t); hs.append(ya); Vs.append(Va); Machs.append(Ma2)
            Drags.append(0.5*ro2*Cd2*Aref*Va**2); alfas.append(alfaa)
            Cds.append(Cd2); ags.append(aga); xs.append(xa)
        t+=DT; step+=1

    return dict(t=np.array(ts), h=np.array(hs), V=np.array(Vs),
                Mach=np.array(Machs), Drag=np.array(Drags),
                alfa=np.array(alfas), Cd=np.array(Cds), ag=np.array(ags),
                x=np.array(xs),
                y_bo=y_bo, V_bo=V_bo, alfa_bo=alfa_bo,
                y_apo=ya, x_apo=xa)

# ── Variante rápida (dt=0.1) para a curva ────────────────────────────────
def apogeu_rapido(ms, tb, C2):
    DT=0.1; f=C2; F0=f*ms*c0/tb
    Aref=PI/4*(d_cm/100)**2
    t=0.0; ya=0.0; xa=0.0; alfaa=alfa0; Va=0.0
    while t<=tb:
        frac=1.0+f*(1.0-t/tb)
        ro=rodiz(ya); Ma=Va/math.sqrt(GAMMA*R*Tz(ya)) if Va>0 else 0
        Cda=splineV(Ma); dP=ATM-pdiz(ya); c=c0*(1.0+Au*dP/F0)
        aga=c*f/(tb*frac*g)-0.5*ro*Va**2*(Aref*Cda/ms/g/frac)-math.sin(alfaa*rad)
        dVdt=aga*g; dadt=-math.cos(alfaa*rad)*g/Va if Va!=0 else 0
        dydt=Va*math.sin(alfaa*rad); dxdt=Va*math.cos(alfaa*rad)
        V1=Va+dVdt*DT; a1=alfaa+dadt*DT; y1=max(ya+dydt*DT,0); x1=xa+dxdt*DT
        ro=rodiz(y1); Ma1=V1/math.sqrt(GAMMA*R*Tz(y1)) if V1>0 else 0
        Cd1=splineV(Ma1); dP=ATM-pdiz(y1); c=c0*(1.0+Au*dP/F0)
        ag1=c*f/(tb*frac*g)-0.5*ro*V1**2*(Aref*Cd1/ms/g/frac)-math.sin(a1*rad)
        dVdt1=ag1*g; dadt1=-math.cos(a1*rad)*g/V1 if V1!=0 else 0
        Va+=0.5*(dVdt+dVdt1)*DT; alfaa+=0.5*(dadt+dadt1)*DT
        ya+=0.5*(dydt+V1*math.sin(a1*rad))*DT; xa+=0.5*(dxdt+V1*math.cos(a1*rad))*DT
        t+=DT
    guard=0
    while Va>=0 and guard<100000:
        ro=rodiz(ya); Ma=Va/math.sqrt(GAMMA*R*Tz(ya)) if Va>0 else 0
        Cda=splineV(Ma); aga=-0.5*ro*Va**2*(Aref*Cda/ms/g)-math.sin(alfaa*rad)
        dVdt=aga*g; dadt=-math.cos(alfaa*rad)*g/Va if Va!=0 else 0
        dydt=Va*math.sin(alfaa*rad)
        V1=Va+dVdt*DT; a1=alfaa+dadt*DT; y1=ya+dydt*DT
        ro=rodiz(y1); Ma1=V1/math.sqrt(GAMMA*R*Tz(y1)) if V1>0 else 0
        Cd1=splineV(Ma1); ag1=-0.5*ro*V1**2*(Aref*Cd1/ms/g)-math.sin(a1*rad)
        dVdt1=ag1*g; dadt1=-math.cos(a1*rad)*g/V1 if V1!=0 else 0
        Va+=0.5*(dVdt+dVdt1)*DT; alfaa+=0.5*(dadt+dadt1)*DT
        ya+=0.5*(dydt+V1*math.sin(a1*rad))*DT
        guard+=1
    return ya

print("Física carregada.")


## 3. Curva de viabilidade T_sl × tb

In [ ]:
H_target = H_target_km * 1000.0
TB_SCAN  = np.concatenate([np.linspace(0.5, 10, 20), np.linspace(10, 60, 30)])

# Com epsilon fixo: C2 = (1-epsilon)/epsilon  (constante)
# Ms escala com Mp: Ms = epsilon * Mp / (1 - epsilon)
C2_eps = (1 - epsilon) / epsilon

curve_Tsl=[]; curve_tb=[]; curve_TW=[]; curve_C2=[]; curve_Mp=[]; curve_Ms=[]

print(f"Calculando curva — H_alvo={H_target_km}km  epsilon={epsilon:.3f}  C2={C2_eps:.3f}  (aguarde...)\n")
print(f"  {'T_sl[kN]':>9}  {'tb[s]':>7}  {'Mp[kg]':>7}  {'Ms[kg]':>7}  {'M0[kg]':>7}  {'T/W':>6}")
print("  " + "-"*55)

for Tsl in np.linspace(Tsl_min_kN, Tsl_max_kN, n_pontos)*1000:
    # Para cada ponto (Tsl, tb): Mp = Tsl*tb/c0, Ms = epsilon*Mp/(1-epsilon)
    # T/W > 1: Tsl > M0*g = Mp/(1-epsilon)*g => tb < c0*(1-epsilon)/g
    tb_tw1 = c0*(1-epsilon)/g * 0.99
    if tb_tw1 <= 1.0: continue

    # Varredura grossa — localiza o PRIMEIRO cruzamento
    tb_lo=tb_hi=apo_prev=None
    for tb in TB_SCAN:
        if tb >= tb_tw1: break
        Mp_t = Tsl*tb/c0
        Ms_t = epsilon*Mp_t/(1-epsilon)
        try:
            apo = apogeu_rapido(Ms_t, tb, C2_eps)
        except Exception:
            break
        if apo_prev is not None and (apo_prev-H_target)*(apo-H_target)<0:
            idx = list(TB_SCAN).index(tb)
            tb_lo=TB_SCAN[idx-1]; tb_hi=tb
            break
        apo_prev = apo

    if tb_lo is None: continue

    try:
        def residuo(tb):
            Mp_t = Tsl*tb/c0
            Ms_t = epsilon*Mp_t/(1-epsilon)
            return apogeu_rapido(Ms_t, tb, C2_eps) - H_target

        tb_sol = brentq(residuo, tb_lo, tb_hi, xtol=0.05)
        Mp_sol = Tsl*tb_sol/c0
        Ms_sol = epsilon*Mp_sol/(1-epsilon)
        M0_sol = Mp_sol + Ms_sol
        TW_sol = Tsl/(M0_sol*g)
        curve_Tsl.append(Tsl/1000); curve_tb.append(tb_sol)
        curve_TW.append(TW_sol); curve_C2.append(C2_eps)
        curve_Mp.append(Mp_sol); curve_Ms.append(Ms_sol)
        print(f"  {Tsl/1000:>9.3f}  {tb_sol:>7.2f}  {Mp_sol:>7.3f}  "
              f"{Ms_sol:>7.3f}  {M0_sol:>7.3f}  {TW_sol:>6.2f}")
    except Exception:
        pass

print(f"\n{len(curve_Tsl)} pontos na curva.")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    f"Curva de viabilidade  |  H_alvo={H_target_km:.0f}km  "
    f"epsilon={epsilon:.2f}  C2={C2_eps:.2f}  d={d_cm:.0f}cm  alfa={alfa0:.0f}°",
    fontsize=12, fontweight='bold'
)
axes[0].plot(curve_tb, curve_Tsl, 'b-o', lw=2, ms=5)
axes[0].set_xlabel('Tempo de queima tb [s]'); axes[0].set_ylabel('T_sl [kN]')
axes[0].set_title('Empuxo × Tempo de queima'); axes[0].grid(True, alpha=0.3)

axes[1].plot(curve_Tsl, curve_TW, 'r-o', lw=2, ms=5)
axes[1].axhline(1.5, color='gray', ls='--', lw=1, label='T/W=1.5')
axes[1].set_xlabel('T_sl [kN]'); axes[1].set_ylabel('T/W')
axes[1].set_title('Razão T/W'); axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

axes[2].plot(curve_Tsl, curve_Mp, 'g-o', lw=2, ms=5, label='Mp')
axes[2].plot(curve_Tsl, curve_Ms, 'b--o', lw=1.5, ms=4, label='Ms')
axes[2].set_xlabel('T_sl [kN]'); axes[2].set_ylabel('Massa [kg]')
axes[2].set_title('Mp e Ms ao longo da curva')
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Escolha um ponto da curva
Olhe a curva acima e defina o `tb_escolhido` desejado.

In [ ]:
tb_escolhido = 25.0    # [s]  ← edite aqui


## 5. Trajetória completa — dt = 0.01 s

In [ ]:
# Deriva parâmetros do tb escolhido (interpolando na curva)
Tsl_escolhido = np.interp(tb_escolhido, curve_tb[::-1], curve_Tsl[::-1]) * 1000
Mp_escolhido  = Tsl_escolhido * tb_escolhido / c0
Ms_escolhido  = epsilon * Mp_escolhido / (1 - epsilon)
M0_escolhido  = Mp_escolhido + Ms_escolhido
C2_escolhido  = (1 - epsilon) / epsilon   # = C2_eps
TW_escolhido  = Tsl_escolhido / (M0_escolhido * g)

print(f"Ponto escolhido — tb = {tb_escolhido} s")
print(f"  T_sl    = {Tsl_escolhido/1000:.4f} kN")
print(f"  Mp      = {Mp_escolhido:.3f} kg")
print(f"  Ms      = {Ms_escolhido:.3f} kg  (epsilon={epsilon:.2f})")
print(f"  M0      = {M0_escolhido:.3f} kg")
print(f"  C2      = {C2_escolhido:.4f}")
print(f"  T/W     = {TW_escolhido:.3f}")
print()

# Trajetória completa dt=0.01s
res = run_trajectory(Ms_escolhido, tb_escolhido, C2_escolhido)

print(f"BURNOUT  h={res['y_bo']/1000:.3f}km   V={res['V_bo']:.2f}m/s")
print(f"APOGEU   h={res['y_apo']/1000:.4f}km  (alvo: {H_target_km}km)")

# Tabela
print()
print(f"{'t[s]':>7}  {'h[m]':>10}  {'V[m/s]':>8}  {'Mach':>7}  "
      f"{'Drag[N]':>9}  {'alfa[°]':>8}  {'Cd':>8}")
print("-"*70)
for i in range(min(30, len(res['t']))):
    print(f"  {res['t'][i]:>5.1f}  {res['h'][i]:>10.2f}  {res['V'][i]:>8.3f}  "
          f"{res['Mach'][i]:>7.4f}  {res['Drag'][i]:>9.2f}  "
          f"{res['alfa'][i]:>8.4f}  {res['Cd'][i]:>8.5f}")
if len(res['t'])>30:
    print(f"  ... ({len(res['t'])} linhas total)")

# Gráficos
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(
    f"Trajetória  |  tb={tb_escolhido}s  T_sl={Tsl_escolhido/1000:.3f}kN  "
    f"Ms={Ms_escolhido:.1f}kg  Mp={Mp_escolhido:.1f}kg  "
    f"Apogeu={res['y_apo']/1000:.2f}km",
    fontsize=11, fontweight='bold'
)
t=res['t']

axes[0,0].plot(t, res['h']/1000, 'b-', lw=1.5)
axes[0,0].axvline(tb_escolhido, color='r', ls='--', lw=1, label=f'Burnout t={tb_escolhido}s')
axes[0,0].axhline(res['y_apo']/1000, color='g', ls=':', lw=1, label=f"Apogeu {res['y_apo']/1000:.2f}km")
axes[0,0].set_xlabel('t [s]'); axes[0,0].set_ylabel('h [km]')
axes[0,0].set_title('Altitude'); axes[0,0].legend(fontsize=8); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(t, res['V'], 'r-', lw=1.5)
axes[0,1].axvline(tb_escolhido, color='r', ls='--', lw=1)
axes[0,1].set_xlabel('t [s]'); axes[0,1].set_ylabel('V [m/s]')
axes[0,1].set_title('Velocidade'); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(t, res['Mach'], 'g-', lw=1.5)
axes[0,2].axvline(tb_escolhido, color='r', ls='--', lw=1)
axes[0,2].axhline(1.0, color='k', ls=':', lw=0.8, label='Mach 1')
axes[0,2].set_xlabel('t [s]'); axes[0,2].set_ylabel('Mach')
axes[0,2].set_title('Mach'); axes[0,2].legend(fontsize=8); axes[0,2].grid(True, alpha=0.3)

axes[1,0].plot(res['x']/1000, res['h']/1000, 'b-', lw=1.5)
axes[1,0].scatter([res['x_apo']/1000],[res['y_apo']/1000], color='g', s=80, marker='*', zorder=5)
axes[1,0].set_xlabel('x [km]'); axes[1,0].set_ylabel('h [km]')
axes[1,0].set_title('Trajetória x-y'); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(t, res['Drag']/1000, 'm-', lw=1.5)
axes[1,1].axvline(tb_escolhido, color='r', ls='--', lw=1)
axes[1,1].set_xlabel('t [s]'); axes[1,1].set_ylabel('Drag [kN]')
axes[1,1].set_title('Arrasto'); axes[1,1].grid(True, alpha=0.3)

axes[1,2].plot(t, res['alfa'], 'c-', lw=1.5)
axes[1,2].axvline(tb_escolhido, color='r', ls='--', lw=1)
axes[1,2].set_xlabel('t [s]'); axes[1,2].set_ylabel('alfa [°]')
axes[1,2].set_title('Ângulo de trajetória'); axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
